# 🚗 Hyundai Motor Company (005380.KS) — Comprehensive Investment Analysis
### Deep Fundamental, Technical & Valuation Analysis
---
*Analysis Date: February 2026 | Currency: KRW (Korean Won) | Exchange: Korea Stock Exchange*

**Analyst Framework:** This notebook performs institutional-grade analysis covering 10-year price history,
quarterly/annual financial statements, profitability metrics, balance sheet health, free cash flow generation,
DCF valuation with sensitivity analysis, and multi-indicator technical analysis — all using Plotly interactive charts.

In [ ]:
# ============================================================
# SETUP & IMPORTS
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_dark'

# Configuration
TICKER = '005380.KS'
COMPANY = 'Hyundai Motor Company'
CURRENCY = 'KRW'
FX_RATE_USD = 1450  # Approximate KRW per USD for context

print(f"📊 Initializing analysis for {COMPANY} ({TICKER})")
print(f"{'='*60}")

In [ ]:
# ============================================================
# DATA RETRIEVAL
# ============================================================
stock = yf.Ticker(TICKER)
info = stock.info

# Price data - 10 years
price_data = stock.history(period='10y')
price_data.index = price_data.index.tz_localize(None)

# Financial statements - Annual
income_stmt = stock.financials
balance_sheet = stock.balance_sheet
cash_flow = stock.cashflow

# Financial statements - Quarterly
q_income = stock.quarterly_financials
q_balance = stock.quarterly_balance_sheet
q_cashflow = stock.quarterly_cashflow

print(f"✅ Price data: {len(price_data)} trading days ({price_data.index[0].date()} to {price_data.index[-1].date()})")
print(f"✅ Annual statements: {len(income_stmt.columns)} years")
print(f"✅ Quarterly statements: {len(q_income.columns)} quarters")

---
## 1. Company Profile & Overview

In [ ]:
# ============================================================
# COMPANY PROFILE
# ============================================================
profile_data = {
    'Company Name': info.get('longName', COMPANY),
    'Sector': info.get('sector', 'N/A'),
    'Industry': info.get('industry', 'N/A'),
    'Country': info.get('country', 'South Korea'),
    'Employees': f"{info.get('fullTimeEmployees', 0):,}",
    'Market Cap (KRW T)': f"₩{info.get('marketCap', 0)/1e12:.1f}T",
    'Market Cap (USD B)': f"${info.get('marketCap', 0)/1e12/FX_RATE_USD*1000:.1f}B",
    'Current Price': f"₩{info.get('currentPrice', info.get('regularMarketPrice', 0)):,.0f}",
    'Enterprise Value (KRW T)': f"₩{info.get('enterpriseValue', 0)/1e12:.1f}T",
    '52-Week High': f"₩{info.get('fiftyTwoWeekHigh', 0):,.0f}",
    '52-Week Low': f"₩{info.get('fiftyTwoWeekLow', 0):,.0f}",
    'Beta': f"{info.get('beta', 'N/A')}",
    'Forward P/E': f"{info.get('forwardPE', 'N/A')}",
    'Dividend Yield': f"{info.get('dividendYield', 0)*100:.2f}%" if info.get('dividendYield') and info.get('dividendYield') < 1 else f"{info.get('dividendYield', 0):.2f}%" if info.get('dividendYield') else 'N/A',
    'Shares Outstanding': f"{info.get('sharesOutstanding', 0)/1e6:.1f}M"
}

print(f"{'='*60}")
print(f"  {COMPANY} — Company Profile")
print(f"{'='*60}")
for k, v in profile_data.items():
    print(f"  {k:<28} {v}")

print(f"\n{'='*60}")
print("  Business Description")
print(f"{'='*60}")
desc = info.get('longBusinessSummary', 'N/A')
# Word wrap
import textwrap
print(textwrap.fill(desc, width=90))

---
## 2. 10-Year Price History & Performance

In [ ]:
# ============================================================
# 10-YEAR PRICE CHART WITH VOLUME
# ============================================================
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    row_heights=[0.75, 0.25],
                    subplot_titles=[f'{COMPANY} — 10-Year Price History', 'Volume'])

# Candlestick chart
fig.add_trace(go.Candlestick(
    x=price_data.index, open=price_data['Open'], high=price_data['High'],
    low=price_data['Low'], close=price_data['Close'], name='Price',
    increasing_line_color='#00d4aa', decreasing_line_color='#ff4757'
), row=1, col=1)

# Moving averages
for period, color, name in [(50, '#ffa502', '50 EMA'), (200, '#ff6348', '200 EMA')]:
    ema = price_data['Close'].ewm(span=period).mean()
    fig.add_trace(go.Scatter(x=price_data.index, y=ema, name=name,
                             line=dict(color=color, width=1.5)), row=1, col=1)

# Volume
colors = ['#00d4aa' if c >= o else '#ff4757' for c, o in zip(price_data['Close'], price_data['Open'])]
fig.add_trace(go.Bar(x=price_data.index, y=price_data['Volume'], name='Volume',
                     marker_color=colors, opacity=0.7), row=2, col=1)

fig.update_layout(height=700, xaxis_rangeslider_visible=False,
                  template='plotly_dark', showlegend=True,
                  legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.update_yaxes(title_text='Price (KRW)', row=1, col=1)
fig.update_yaxes(title_text='Volume', row=2, col=1)
fig.show()

In [ ]:
# ============================================================
# PERFORMANCE METRICS
# ============================================================
current_price = price_data['Close'].iloc[-1]
periods = {
    '1 Month': 21, '3 Months': 63, '6 Months': 126,
    'YTD': len(price_data[price_data.index >= f'{datetime.now().year}-01-01']),
    '1 Year': 252, '3 Years': 756, '5 Years': 1260, '10 Years': len(price_data)
}

print(f"{'='*60}")
print(f"  Price Performance Summary")
print(f"{'='*60}")
print(f"  {'Period':<15} {'Return':>12} {'Start Price':>15} {'End Price':>15}")
print(f"  {'-'*57}")

for label, days in periods.items():
    if days > 0 and days <= len(price_data):
        start_p = price_data['Close'].iloc[-min(days, len(price_data))]
        ret = (current_price / start_p - 1) * 100
        print(f"  {label:<15} {ret:>+11.2f}% {start_p:>14,.0f} {current_price:>14,.0f}")

# Annualised return & volatility
daily_returns = price_data['Close'].pct_change().dropna()
ann_return = daily_returns.mean() * 252 * 100
ann_vol = daily_returns.std() * np.sqrt(252) * 100
sharpe = ann_return / ann_vol if ann_vol > 0 else 0
max_dd = ((price_data['Close'] / price_data['Close'].cummax()) - 1).min() * 100

print(f"\n  {'Annualised Return':<28} {ann_return:>+.2f}%")
print(f"  {'Annualised Volatility':<28} {ann_vol:.2f}%")
print(f"  {'Sharpe Ratio (RF=0)':<28} {sharpe:.3f}")
print(f"  {'Maximum Drawdown':<28} {max_dd:.2f}%")

In [ ]:
# ============================================================
# ANNUAL RETURNS HEATMAP
# ============================================================
monthly_returns = price_data['Close'].resample('ME').last().pct_change().dropna()
monthly_df = pd.DataFrame({'Return': monthly_returns.values * 100}, index=monthly_returns.index)
monthly_df['Year'] = monthly_df.index.year
monthly_df['Month'] = monthly_df.index.month

pivot = monthly_df.pivot_table(values='Return', index='Year', columns='Month', aggfunc='mean')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = go.Figure(data=go.Heatmap(
    z=pivot.values, x=pivot.columns, y=pivot.index.astype(str),
    colorscale='RdYlGn', zmid=0, text=np.round(pivot.values, 1),
    texttemplate='%{text:.1f}%', textfont={'size': 10},
    colorbar=dict(title='Return %')
))
fig.update_layout(title=f'{COMPANY} — Monthly Returns Heatmap (%)',
                  height=450, template='plotly_dark')
fig.show()

---
## 3. Revenue & Profitability Analysis

In [ ]:
# ============================================================
# ANNUAL INCOME STATEMENT ANALYSIS
# ============================================================
def safe_get(df, row, col=None):
    """Safely extract value from DataFrame"""
    try:
        if col is not None:
            return df.loc[row, col] if row in df.index else np.nan
        return df.loc[row] if row in df.index else pd.Series(dtype=float)
    except:
        return np.nan

# Build annual summary
annual_cols = sorted(income_stmt.columns)
annual_data = pd.DataFrame(index=[c.year for c in annual_cols])

for col in annual_cols:
    yr = col.year
    annual_data.loc[yr, 'Revenue'] = safe_get(income_stmt, 'Total Revenue', col)
    annual_data.loc[yr, 'Cost of Revenue'] = safe_get(income_stmt, 'Cost Of Revenue', col)
    annual_data.loc[yr, 'Gross Profit'] = safe_get(income_stmt, 'Gross Profit', col)
    annual_data.loc[yr, 'Operating Income'] = safe_get(income_stmt, 'Operating Income', col)
    annual_data.loc[yr, 'EBITDA'] = safe_get(income_stmt, 'EBITDA', col)
    annual_data.loc[yr, 'EBIT'] = safe_get(income_stmt, 'EBIT', col)
    annual_data.loc[yr, 'Net Income'] = safe_get(income_stmt, 'Net Income', col)
    annual_data.loc[yr, 'R&D'] = safe_get(income_stmt, 'Research And Development', col)
    annual_data.loc[yr, 'SGA'] = safe_get(income_stmt, 'Selling General And Administration', col)
    annual_data.loc[yr, 'Interest Expense'] = safe_get(income_stmt, 'Interest Expense', col)
    annual_data.loc[yr, 'EPS Diluted'] = safe_get(income_stmt, 'Diluted EPS', col)

annual_data = annual_data.sort_index()

# Convert to trillions for display
T = 1e12
display_df = annual_data.copy()
for c in ['Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Income', 'EBITDA', 'EBIT', 'Net Income', 'R&D', 'SGA', 'Interest Expense']:
    if c in display_df.columns:
        display_df[c] = display_df[c] / T

print(f"{'='*80}")
print(f"  Annual Income Statement (KRW Trillions)")
print(f"{'='*80}")
print(display_df[['Revenue', 'Gross Profit', 'Operating Income', 'EBITDA', 'Net Income', 'R&D']].round(2).to_string())

# Margins
print(f"\n{'='*80}")
print(f"  Profitability Margins")
print(f"{'='*80}")
margins = pd.DataFrame(index=annual_data.index)
margins['Gross Margin'] = (annual_data['Gross Profit'] / annual_data['Revenue'] * 100).round(2)
margins['Operating Margin'] = (annual_data['Operating Income'] / annual_data['Revenue'] * 100).round(2)
margins['EBITDA Margin'] = (annual_data['EBITDA'] / annual_data['Revenue'] * 100).round(2)
margins['Net Margin'] = (annual_data['Net Income'] / annual_data['Revenue'] * 100).round(2)
margins['R&D / Revenue'] = (annual_data['R&D'] / annual_data['Revenue'] * 100).round(2)
print(margins.to_string())

In [ ]:
# ============================================================
# REVENUE & PROFIT VISUALIZATION
# ============================================================
fig = make_subplots(rows=1, cols=2, subplot_titles=['Revenue & Profit (KRW T)', 'Profitability Margins (%)'],
                    horizontal_spacing=0.12)

years = annual_data.index.astype(str)

# Revenue & Profits
fig.add_trace(go.Bar(x=years, y=annual_data['Revenue']/T, name='Revenue',
                     marker_color='#4ecdc4', opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(x=years, y=annual_data['Gross Profit']/T, name='Gross Profit',
                     marker_color='#45b7d1', opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(x=years, y=annual_data['Operating Income']/T, name='Operating Income',
                     marker_color='#f9ca24', opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(x=years, y=annual_data['Net Income']/T, name='Net Income',
                     marker_color='#00d4aa', opacity=0.8), row=1, col=1)

# Margins
for col, color in [('Gross Margin', '#4ecdc4'), ('Operating Margin', '#f9ca24'),
                    ('EBITDA Margin', '#45b7d1'), ('Net Margin', '#00d4aa')]:
    fig.add_trace(go.Scatter(x=years, y=margins[col], name=col, mode='lines+markers',
                             line=dict(color=color, width=2.5),
                             marker=dict(size=8)), row=1, col=2)

fig.update_layout(height=500, template='plotly_dark', barmode='group',
                  title=f'{COMPANY} — Revenue, Profits & Margins',
                  legend=dict(orientation='h', yanchor='bottom', y=-0.25))
fig.update_yaxes(title_text='KRW Trillions', row=1, col=1)
fig.update_yaxes(title_text='%', row=1, col=2)
fig.show()

In [ ]:
# ============================================================
# QUARTERLY REVENUE & EARNINGS TREND
# ============================================================
q_cols = sorted(q_income.columns)
q_data = pd.DataFrame()

for col in q_cols:
    label = col.strftime('%Y-Q%m').replace('Q03','Q1').replace('Q06','Q2').replace('Q09','Q3').replace('Q12','Q4')
    # Map month to quarter
    m = col.month
    q_label = f"{col.year}-Q{(m-1)//3+1}"
    q_data.loc[q_label, 'Revenue'] = safe_get(q_income, 'Total Revenue', col)
    q_data.loc[q_label, 'Operating Income'] = safe_get(q_income, 'Operating Income', col)
    q_data.loc[q_label, 'Net Income'] = safe_get(q_income, 'Net Income', col)
    q_data.loc[q_label, 'EBITDA'] = safe_get(q_income, 'EBITDA', col)
    q_data.loc[q_label, 'Gross Profit'] = safe_get(q_income, 'Gross Profit', col)

q_data = q_data.sort_index()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['Quarterly Revenue (KRW T)', 'Quarterly Net Income (KRW T)'])

fig.add_trace(go.Bar(x=q_data.index, y=q_data['Revenue']/T, name='Revenue',
                     marker_color='#4ecdc4'), row=1, col=1)
fig.add_trace(go.Bar(x=q_data.index, y=q_data['Net Income']/T, name='Net Income',
                     marker_color='#00d4aa'), row=2, col=1)

# Add operating income overlay
fig.add_trace(go.Scatter(x=q_data.index, y=q_data['Operating Income']/T, name='Operating Income',
                         mode='lines+markers', line=dict(color='#f9ca24', width=2)), row=2, col=1)

fig.update_layout(height=550, template='plotly_dark',
                  title=f'{COMPANY} — Quarterly Revenue & Earnings Progression',
                  legend=dict(orientation='h', yanchor='bottom', y=-0.15))
fig.show()

# Quarterly margins
print(f"\n{'='*70}")
print(f"  Quarterly Margins")
print(f"{'='*70}")
q_margins = pd.DataFrame(index=q_data.index)
q_margins['Gross Margin %'] = (q_data['Gross Profit'] / q_data['Revenue'] * 100).round(2)
q_margins['Operating Margin %'] = (q_data['Operating Income'] / q_data['Revenue'] * 100).round(2)
q_margins['Net Margin %'] = (q_data['Net Income'] / q_data['Revenue'] * 100).round(2)
print(q_margins.to_string())

In [ ]:
# ============================================================
# COST STRUCTURE ANALYSIS (REVENUE WATERFALL)
# ============================================================
latest_yr = annual_data.index[-1]
rev = annual_data.loc[latest_yr, 'Revenue']
cogs = annual_data.loc[latest_yr, 'Cost of Revenue']
gp = annual_data.loc[latest_yr, 'Gross Profit']
rd = annual_data.loc[latest_yr, 'R&D'] if not pd.isna(annual_data.loc[latest_yr, 'R&D']) else 0
sga = annual_data.loc[latest_yr, 'SGA'] if not pd.isna(annual_data.loc[latest_yr, 'SGA']) else 0
oi = annual_data.loc[latest_yr, 'Operating Income']
ni = annual_data.loc[latest_yr, 'Net Income']

fig = go.Figure(go.Waterfall(
    x=['Revenue', 'COGS', 'Gross Profit', 'R&D', 'SG&A', 'Other OpEx', 'Operating Income', 'Below Line', 'Net Income'],
    y=[rev/T, -cogs/T, 0, -rd/T, -sga/T, -(gp-rd-sga-oi)/T, 0, -(oi-ni)/T, 0],
    measure=['absolute', 'relative', 'total', 'relative', 'relative', 'relative', 'total', 'relative', 'total'],
    connector={'line': {'color': 'rgba(255,255,255,0.3)'}},
    increasing={'marker': {'color': '#00d4aa'}},
    decreasing={'marker': {'color': '#ff4757'}},
    totals={'marker': {'color': '#4ecdc4'}},
    text=[f'₩{abs(v)/T:.1f}T' for v in [rev, cogs, gp, rd, sga, gp-rd-sga-oi, oi, oi-ni, ni]],
    textposition='outside'
))
fig.update_layout(title=f'{COMPANY} — Revenue to Net Income Waterfall ({latest_yr})',
                  height=500, template='plotly_dark', yaxis_title='KRW Trillions')
fig.show()

---
## 4. Balance Sheet Analysis

In [ ]:
# ============================================================
# BALANCE SHEET DEEP DIVE
# ============================================================
bs_cols = sorted(balance_sheet.columns)
bs_data = pd.DataFrame(index=[c.year for c in bs_cols])

for col in bs_cols:
    yr = col.year
    bs_data.loc[yr, 'Total Assets'] = safe_get(balance_sheet, 'Total Assets', col)
    bs_data.loc[yr, 'Current Assets'] = safe_get(balance_sheet, 'Current Assets', col)
    bs_data.loc[yr, 'Cash & Equivalents'] = safe_get(balance_sheet, 'Cash And Cash Equivalents', col)
    bs_data.loc[yr, 'Short Term Investments'] = safe_get(balance_sheet, 'Other Short Term Investments', col)
    bs_data.loc[yr, 'Inventory'] = safe_get(balance_sheet, 'Inventory', col)
    bs_data.loc[yr, 'Accounts Receivable'] = safe_get(balance_sheet, 'Accounts Receivable', col)
    bs_data.loc[yr, 'Net PPE'] = safe_get(balance_sheet, 'Net PPE', col)
    bs_data.loc[yr, 'Total Non Current Assets'] = safe_get(balance_sheet, 'Total Non Current Assets', col)
    bs_data.loc[yr, 'Goodwill'] = safe_get(balance_sheet, 'Goodwill', col)
    bs_data.loc[yr, 'Current Liabilities'] = safe_get(balance_sheet, 'Current Liabilities', col)
    bs_data.loc[yr, 'Total Liabilities'] = safe_get(balance_sheet, 'Total Liabilities Net Minority Interest', col)
    bs_data.loc[yr, 'Long Term Debt'] = safe_get(balance_sheet, 'Long Term Debt', col)
    bs_data.loc[yr, 'Current Debt'] = safe_get(balance_sheet, 'Current Debt', col)
    bs_data.loc[yr, 'Total Debt'] = safe_get(balance_sheet, 'Total Debt', col)
    bs_data.loc[yr, 'Stockholders Equity'] = safe_get(balance_sheet, 'Stockholders Equity', col)
    bs_data.loc[yr, 'Retained Earnings'] = safe_get(balance_sheet, 'Retained Earnings', col)
    bs_data.loc[yr, 'Minority Interest'] = safe_get(balance_sheet, 'Minority Interest', col)
    bs_data.loc[yr, 'Working Capital'] = safe_get(balance_sheet, 'Working Capital', col)
    bs_data.loc[yr, 'Net Debt'] = safe_get(balance_sheet, 'Net Debt', col)
    bs_data.loc[yr, 'Tangible Book Value'] = safe_get(balance_sheet, 'Tangible Book Value', col)
    bs_data.loc[yr, 'Invested Capital'] = safe_get(balance_sheet, 'Invested Capital', col)

bs_data = bs_data.sort_index()

print(f"{'='*80}")
print(f"  Balance Sheet Summary (KRW Trillions)")
print(f"{'='*80}")
bs_display = (bs_data[['Total Assets', 'Current Assets', 'Cash & Equivalents', 'Inventory',
                        'Net PPE', 'Total Liabilities', 'Total Debt', 'Long Term Debt',
                        'Stockholders Equity', 'Working Capital', 'Net Debt', 'Invested Capital']] / T).round(2)
print(bs_display.to_string())

In [ ]:
# ============================================================
# BALANCE SHEET RATIOS
# ============================================================
bs_ratios = pd.DataFrame(index=bs_data.index)
bs_ratios['Current Ratio'] = (bs_data['Current Assets'] / bs_data['Current Liabilities']).round(2)
bs_ratios['Debt to Equity'] = (bs_data['Total Debt'] / bs_data['Stockholders Equity']).round(2)
bs_ratios['Debt to Assets'] = (bs_data['Total Debt'] / bs_data['Total Assets']).round(2)
bs_ratios['Net Debt to EBITDA'] = (bs_data['Net Debt'] / annual_data['EBITDA']).round(2)
bs_ratios['Equity Ratio'] = (bs_data['Stockholders Equity'] / bs_data['Total Assets'] * 100).round(1)
bs_ratios['Asset Turnover'] = (annual_data['Revenue'] / bs_data['Total Assets']).round(2)

# ROE and ROA
bs_ratios['ROE %'] = (annual_data['Net Income'] / bs_data['Stockholders Equity'] * 100).round(2)
bs_ratios['ROA %'] = (annual_data['Net Income'] / bs_data['Total Assets'] * 100).round(2)
bs_ratios['ROIC %'] = (annual_data['EBIT'] * (1 - 0.25) / bs_data['Invested Capital'] * 100).round(2)

print(f"{'='*80}")
print(f"  Key Financial Ratios")
print(f"{'='*80}")
print(bs_ratios.to_string())

In [ ]:
# ============================================================
# BALANCE SHEET COMPOSITION CHART
# ============================================================
fig = make_subplots(rows=1, cols=2, subplot_titles=['Asset Composition', 'Capital Structure'],
                    horizontal_spacing=0.12)

years_str = bs_data.index.astype(str)

# Assets
fig.add_trace(go.Bar(x=years_str, y=bs_data['Cash & Equivalents']/T, name='Cash',
                     marker_color='#00d4aa'), row=1, col=1)
fig.add_trace(go.Bar(x=years_str, y=bs_data['Inventory']/T, name='Inventory',
                     marker_color='#ffa502'), row=1, col=1)
fig.add_trace(go.Bar(x=years_str, y=bs_data['Accounts Receivable']/T, name='Receivables',
                     marker_color='#45b7d1'), row=1, col=1)
fig.add_trace(go.Bar(x=years_str, y=bs_data['Net PPE']/T, name='Net PP&E',
                     marker_color='#ff6348'), row=1, col=1)
other_assets = (bs_data['Total Assets'] - bs_data['Cash & Equivalents'].fillna(0) - bs_data['Inventory'].fillna(0) - bs_data['Accounts Receivable'].fillna(0) - bs_data['Net PPE'].fillna(0))
fig.add_trace(go.Bar(x=years_str, y=other_assets/T, name='Other Assets',
                     marker_color='#a29bfe'), row=1, col=1)

# Capital Structure
fig.add_trace(go.Bar(x=years_str, y=bs_data['Current Liabilities']/T, name='Current Liabilities',
                     marker_color='#ff4757'), row=1, col=2)
lt_liab = bs_data['Total Liabilities'] - bs_data['Current Liabilities']
fig.add_trace(go.Bar(x=years_str, y=lt_liab/T, name='Long-Term Liabilities',
                     marker_color='#ff6348'), row=1, col=2)
fig.add_trace(go.Bar(x=years_str, y=bs_data['Stockholders Equity']/T, name='Equity',
                     marker_color='#00d4aa'), row=1, col=2)

fig.update_layout(height=500, template='plotly_dark', barmode='stack',
                  title=f'{COMPANY} — Balance Sheet Composition (KRW T)',
                  legend=dict(orientation='h', yanchor='bottom', y=-0.25))
fig.show()

In [ ]:
# ============================================================
# RETURN ON EQUITY DECOMPOSITION (DuPont Analysis)
# ============================================================
dupont = pd.DataFrame(index=annual_data.index)
dupont['Net Margin (%)'] = (annual_data['Net Income'] / annual_data['Revenue'] * 100).round(2)
dupont['Asset Turnover'] = (annual_data['Revenue'] / bs_data['Total Assets']).round(3)
dupont['Equity Multiplier'] = (bs_data['Total Assets'] / bs_data['Stockholders Equity']).round(2)
dupont['ROE (%)'] = (dupont['Net Margin (%)'] * dupont['Asset Turnover'] * dupont['Equity Multiplier'] / 100).round(2)

print(f"{'='*65}")
print(f"  DuPont ROE Decomposition")
print(f"  ROE = Net Margin × Asset Turnover × Equity Multiplier")
print(f"{'='*65}")
print(dupont.to_string())

fig = make_subplots(rows=1, cols=1)
for col, color in [('Net Margin (%)', '#00d4aa'), ('Asset Turnover', '#ffa502'), ('Equity Multiplier', '#ff6348')]:
    fig.add_trace(go.Scatter(x=dupont.index.astype(str), y=dupont[col], name=col,
                             mode='lines+markers', line=dict(color=color, width=2.5),
                             marker=dict(size=10)))
fig.update_layout(title=f'{COMPANY} — DuPont Analysis Components', height=400, template='plotly_dark')
fig.show()

---
## 5. Free Cash Flow Analysis

In [ ]:
# ============================================================
# CASH FLOW STATEMENT ANALYSIS
# ============================================================
cf_cols = sorted(cash_flow.columns)
cf_data = pd.DataFrame(index=[c.year for c in cf_cols])

for col in cf_cols:
    yr = col.year
    cf_data.loc[yr, 'Operating CF'] = safe_get(cash_flow, 'Operating Cash Flow', col)
    cf_data.loc[yr, 'CapEx'] = safe_get(cash_flow, 'Capital Expenditure', col)
    cf_data.loc[yr, 'Free Cash Flow'] = safe_get(cash_flow, 'Free Cash Flow', col)
    cf_data.loc[yr, 'Investing CF'] = safe_get(cash_flow, 'Investing Cash Flow', col)
    cf_data.loc[yr, 'Financing CF'] = safe_get(cash_flow, 'Financing Cash Flow', col)
    cf_data.loc[yr, 'Dividends Paid'] = safe_get(cash_flow, 'Cash Dividends Paid', col)
    cf_data.loc[yr, 'Buybacks'] = safe_get(cash_flow, 'Repurchase Of Capital Stock', col)
    cf_data.loc[yr, 'D&A'] = safe_get(cash_flow, 'Depreciation And Amortization', col)
    cf_data.loc[yr, 'Debt Issuance'] = safe_get(cash_flow, 'Issuance Of Debt', col)
    cf_data.loc[yr, 'Debt Repayment'] = safe_get(cash_flow, 'Repayment Of Debt', col)

cf_data = cf_data.sort_index()

# Add derived metrics
cf_data['FCF Margin %'] = (cf_data['Free Cash Flow'] / annual_data['Revenue'] * 100).round(2)
cf_data['CapEx / Revenue %'] = (abs(cf_data['CapEx']) / annual_data['Revenue'] * 100).round(2)
cf_data['OCF / Net Income'] = (cf_data['Operating CF'] / annual_data['Net Income']).round(2)
cf_data['FCF / Net Income'] = (cf_data['Free Cash Flow'] / annual_data['Net Income']).round(2)

# FCF Yield
shares = info.get('sharesOutstanding', 1)
cf_data['FCF per Share'] = (cf_data['Free Cash Flow'] / shares).round(0)

print(f"{'='*80}")
print(f"  Cash Flow Statement Summary (KRW Trillions)")
print(f"{'='*80}")
cf_display = cf_data[['Operating CF', 'CapEx', 'Free Cash Flow', 'Investing CF', 'Financing CF', 'Dividends Paid']].copy()
for c in cf_display.columns:
    cf_display[c] = (cf_display[c] / T).round(2)
print(cf_display.to_string())

print(f"\n{'='*80}")
print(f"  Cash Flow Quality Metrics")
print(f"{'='*80}")
print(cf_data[['FCF Margin %', 'CapEx / Revenue %', 'OCF / Net Income', 'FCF / Net Income', 'FCF per Share']].to_string())

In [ ]:
# ============================================================
# CASH FLOW VISUALIZATION
# ============================================================
fig = make_subplots(rows=1, cols=2, subplot_titles=['Cash Flow Components (KRW T)', 'FCF Margin & CapEx Intensity'],
                    horizontal_spacing=0.12)

years_str = cf_data.index.astype(str)

fig.add_trace(go.Bar(x=years_str, y=cf_data['Operating CF']/T, name='Operating CF',
                     marker_color='#00d4aa'), row=1, col=1)
fig.add_trace(go.Bar(x=years_str, y=cf_data['CapEx']/T, name='CapEx (neg)',
                     marker_color='#ff4757'), row=1, col=1)
fig.add_trace(go.Bar(x=years_str, y=cf_data['Free Cash Flow']/T, name='Free Cash Flow',
                     marker_color='#4ecdc4'), row=1, col=1)

# FCF margins
fig.add_trace(go.Scatter(x=years_str, y=cf_data['FCF Margin %'], name='FCF Margin %',
                         mode='lines+markers', line=dict(color='#00d4aa', width=3),
                         marker=dict(size=10)), row=1, col=2)
fig.add_trace(go.Scatter(x=years_str, y=cf_data['CapEx / Revenue %'], name='CapEx / Rev %',
                         mode='lines+markers', line=dict(color='#ff6348', width=3),
                         marker=dict(size=10)), row=1, col=2)

fig.update_layout(height=500, template='plotly_dark', barmode='group',
                  title=f'{COMPANY} — Cash Flow Analysis',
                  legend=dict(orientation='h', yanchor='bottom', y=-0.25))
fig.show()

---
## 6. Valuation — DCF Model & Multiples

In [ ]:
# ============================================================
# DCF VALUATION MODEL
# ============================================================

# Get latest FCF and financials
latest_fcf = cf_data['Free Cash Flow'].iloc[-1]
latest_revenue = annual_data['Revenue'].iloc[-1]
latest_ebitda = annual_data['EBITDA'].iloc[-1]
latest_net_income = annual_data['Net Income'].iloc[-1]
total_debt = bs_data['Total Debt'].iloc[-1]
cash = bs_data['Cash & Equivalents'].iloc[-1]
equity_value = bs_data['Stockholders Equity'].iloc[-1]
shares_out = info.get('sharesOutstanding', 202006914)
current_price_val = info.get('currentPrice', info.get('regularMarketPrice', 523000))
market_cap = current_price_val * shares_out
ev = market_cap + total_debt - cash
beta = info.get('beta', 1.32)

# Revenue growth rate (historical CAGR)
rev_first = annual_data['Revenue'].iloc[0]
rev_last = annual_data['Revenue'].iloc[-1]
n_years = len(annual_data) - 1
rev_cagr = (rev_last / rev_first) ** (1/n_years) - 1 if n_years > 0 else 0.05

# WACC Estimation
risk_free_rate = 0.035     # Korean 10Y government bond
equity_risk_premium = 0.06 # Korea equity risk premium
cost_of_equity = risk_free_rate + beta * equity_risk_premium
cost_of_debt = 0.04        # Estimated pre-tax cost of debt
tax_rate = 0.25            # Effective tax rate
debt_weight = total_debt / (market_cap + total_debt)
equity_weight = market_cap / (market_cap + total_debt)
wacc = equity_weight * cost_of_equity + debt_weight * cost_of_debt * (1 - tax_rate)

print(f"{'='*65}")
print(f"  DCF Valuation — Key Inputs")
print(f"{'='*65}")
print(f"  Latest FCF:              ₩{latest_fcf/T:.2f}T")
print(f"  Revenue CAGR ({n_years}yr):    {rev_cagr*100:.1f}%")
print(f"  Beta:                    {beta:.2f}")
print(f"  Cost of Equity:          {cost_of_equity*100:.2f}%")
print(f"  Cost of Debt (pre-tax):  {cost_of_debt*100:.2f}%")
print(f"  WACC:                    {wacc*100:.2f}%")
print(f"  Debt Weight:             {debt_weight*100:.1f}%")
print(f"  Equity Weight:           {equity_weight*100:.1f}%")
print(f"  Shares Outstanding:      {shares_out/1e6:.1f}M")
print(f"  Current Market Cap:      ₩{market_cap/T:.1f}T")
print(f"  Enterprise Value:        ₩{ev/T:.1f}T")

# Critical note
if latest_fcf < 0:
    print(f'\n  ⚠️  CRITICAL NOTE: Latest FCF is NEGATIVE (₩{latest_fcf/T:.1f}T)')
    print(f'  Hyundai\'s consolidated statements include Hyundai Capital (financial')
    print(f'  services). Massive investing outflows from auto loan originations distort')
    print(f'  traditional FCF. The DCF below will produce unreliable results.')
    print(f'  → Multiples-based valuation (P/E, EV/EBITDA, P/B) is more appropriate.')
    print(f'  → Sum-of-the-parts (SOTP) analysis separating auto and finance is ideal.')

In [ ]:
# ============================================================
# DCF PROJECTION — 10-YEAR MODEL
# ============================================================
def dcf_model(base_fcf, growth_rates, terminal_growth, discount_rate, total_debt, cash, shares):
    """
    Multi-stage DCF with explicit projection period and Gordon Growth Terminal Value
    """
    projected_fcf = []
    fcf = base_fcf
    
    for i, g in enumerate(growth_rates):
        fcf = fcf * (1 + g)
        projected_fcf.append(fcf)
    
    # Terminal Value (Gordon Growth Model)
    terminal_fcf = projected_fcf[-1] * (1 + terminal_growth)
    terminal_value = terminal_fcf / (discount_rate - terminal_growth)
    
    # Discount all cash flows
    pv_fcfs = []
    for i, f in enumerate(projected_fcf):
        pv = f / (1 + discount_rate) ** (i + 1)
        pv_fcfs.append(pv)
    
    pv_terminal = terminal_value / (1 + discount_rate) ** len(projected_fcf)
    
    enterprise_value = sum(pv_fcfs) + pv_terminal
    equity_value = enterprise_value - total_debt + cash
    value_per_share = equity_value / shares
    
    return {
        'projected_fcf': projected_fcf,
        'pv_fcfs': pv_fcfs,
        'terminal_value': terminal_value,
        'pv_terminal': pv_terminal,
        'sum_pv_fcfs': sum(pv_fcfs),
        'enterprise_value': enterprise_value,
        'equity_value': equity_value,
        'value_per_share': value_per_share
    }

# Growth assumptions: Phase 1 (Years 1-3) higher growth, Phase 2 (Years 4-7) moderate, Phase 3 (Years 8-10) mature
phase1_growth = min(max(rev_cagr, 0.04), 0.12)  # 4-12% based on historical
phase2_growth = phase1_growth * 0.7
phase3_growth = phase2_growth * 0.6
terminal_growth_rate = 0.025  # 2.5% perpetual

growth_rates = [phase1_growth]*3 + [phase2_growth]*4 + [phase3_growth]*3

# Base case
base_result = dcf_model(latest_fcf, growth_rates, terminal_growth_rate, wacc, total_debt, cash, shares_out)

print(f"{'='*70}")
print(f"  DCF Valuation — 10-Year Projection")
print(f"{'='*70}")
print(f"  Growth Assumptions:")
print(f"    Phase 1 (Yr 1-3):  {phase1_growth*100:.1f}%")
print(f"    Phase 2 (Yr 4-7):  {phase2_growth*100:.1f}%")
print(f"    Phase 3 (Yr 8-10): {phase3_growth*100:.1f}%")
print(f"    Terminal Growth:    {terminal_growth_rate*100:.1f}%")
print(f"    WACC:              {wacc*100:.2f}%")

print(f"\n  {'Year':<8} {'Projected FCF':>18} {'PV of FCF':>18}")
print(f"  {'-'*46}")
for i in range(10):
    yr = datetime.now().year + i + 1
    print(f"  {yr:<8} ₩{base_result['projected_fcf'][i]/T:>14.2f}T  ₩{base_result['pv_fcfs'][i]/T:>14.2f}T")

print(f"\n  {'='*50}")
print(f"  Sum of PV(FCFs):       ₩{base_result['sum_pv_fcfs']/T:.2f}T")
print(f"  PV of Terminal Value:  ₩{base_result['pv_terminal']/T:.2f}T")
print(f"  Terminal % of Total:   {base_result['pv_terminal']/base_result['enterprise_value']*100:.1f}%")
print(f"  Enterprise Value:      ₩{base_result['enterprise_value']/T:.2f}T")
print(f"  Less: Total Debt:      ₩{total_debt/T:.2f}T")
print(f"  Plus: Cash:            ₩{cash/T:.2f}T")
print(f"  Equity Value:          ₩{base_result['equity_value']/T:.2f}T")
print(f"  Shares Outstanding:    {shares_out/1e6:.1f}M")
print(f"  {'='*50}")
print(f"  ⭐ Intrinsic Value/Share: ₩{base_result['value_per_share']:,.0f}")
print(f"  📊 Current Price:        ₩{current_price_val:,.0f}")
upside = (base_result['value_per_share'] / current_price_val - 1) * 100
print(f"  {'🟢' if upside > 0 else '🔴'} Upside/Downside:      {upside:+.1f}%")

In [ ]:
# ============================================================
# DCF SENSITIVITY ANALYSIS
# ============================================================
wacc_range = [wacc - 0.02, wacc - 0.01, wacc, wacc + 0.01, wacc + 0.02]
tg_range = [0.015, 0.02, 0.025, 0.03, 0.035]

sensitivity = pd.DataFrame(index=[f'{w*100:.1f}%' for w in wacc_range],
                           columns=[f'{g*100:.1f}%' for g in tg_range])

for i, w in enumerate(wacc_range):
    for j, tg in enumerate(tg_range):
        if w > tg:  # WACC must exceed terminal growth
            result = dcf_model(latest_fcf, growth_rates, tg, w, total_debt, cash, shares_out)
            sensitivity.iloc[i, j] = f"₩{result['value_per_share']:,.0f}"
        else:
            sensitivity.iloc[i, j] = 'N/A'

sensitivity.index.name = 'WACC ↓ / Terminal Growth →'
print(f"{'='*80}")
print(f"  DCF Sensitivity Analysis — Intrinsic Value per Share")
print(f"  Current Price: ₩{current_price_val:,.0f}")
print(f"{'='*80}")
print(sensitivity.to_string())

# Create heatmap
sens_values = pd.DataFrame(index=wacc_range, columns=tg_range)
for i, w in enumerate(wacc_range):
    for j, tg in enumerate(tg_range):
        if w > tg:
            result = dcf_model(latest_fcf, growth_rates, tg, w, total_debt, cash, shares_out)
            sens_values.iloc[i, j] = result['value_per_share']
        else:
            sens_values.iloc[i, j] = np.nan

fig = go.Figure(data=go.Heatmap(
    z=sens_values.values.astype(float),
    x=[f'TG: {g*100:.1f}%' for g in tg_range],
    y=[f'WACC: {w*100:.1f}%' for w in wacc_range],
    colorscale='RdYlGn', zmid=current_price_val,
    text=[[f'₩{v:,.0f}' if not np.isnan(v) else 'N/A' for v in row] for row in sens_values.values.astype(float)],
    texttemplate='%{text}', textfont={'size': 12},
    colorbar=dict(title='Value/Share')
))
fig.update_layout(title=f'{COMPANY} — DCF Sensitivity (Current: ₩{current_price_val:,.0f})',
                  height=400, template='plotly_dark')
fig.show()

In [ ]:
# ============================================================
# MULTIPLES-BASED VALUATION
# ============================================================
print(f"{'='*70}")
print(f"  Multiples-Based Valuation")
print(f"{'='*70}")

pe_ratio = market_cap / latest_net_income if latest_net_income > 0 else None
ev_ebitda = ev / latest_ebitda if latest_ebitda > 0 else None
ev_revenue = ev / latest_revenue if latest_revenue > 0 else None
pb_ratio = market_cap / equity_value if equity_value > 0 else None
fcf_yield = latest_fcf / market_cap * 100 if market_cap > 0 else None
div_yield = info.get('dividendYield', 0)
if div_yield and div_yield > 1:  # yfinance returned percentage not decimal
    div_yield_pct = div_yield
elif div_yield:
    div_yield_pct = div_yield * 100
else:
    div_yield_pct = 0
earnings_yield = (1/pe_ratio * 100) if pe_ratio else None

multiples = {
    'P/E Ratio': f'{pe_ratio:.2f}' if pe_ratio else 'N/A',
    'Forward P/E': f"{info.get('forwardPE', 'N/A')}",
    'EV/EBITDA': f'{ev_ebitda:.2f}' if ev_ebitda else 'N/A',
    'EV/Revenue': f'{ev_revenue:.2f}' if ev_revenue else 'N/A',
    'P/B Ratio': f'{pb_ratio:.2f}' if pb_ratio else 'N/A',
    'FCF Yield': f'{fcf_yield:.2f}%' if fcf_yield else 'N/A',
    'Earnings Yield': f'{earnings_yield:.2f}%' if earnings_yield else 'N/A',
    'Dividend Yield': f'{div_yield_pct:.2f}%',
}

# Auto industry peer averages (approximate)
peer_avg = {
    'P/E Ratio': '8-12',
    'Forward P/E': '7-11',
    'EV/EBITDA': '4-7',
    'EV/Revenue': '0.3-0.8',
    'P/B Ratio': '0.8-1.5',
    'FCF Yield': '4-8%',
    'Earnings Yield': '8-15%',
    'Dividend Yield': '2-4%',
}

print(f"  {'Metric':<20} {'Hyundai':>15} {'Auto Industry Avg':>20}")
print(f"  {'-'*55}")
for k in multiples:
    print(f"  {k:<20} {multiples[k]:>15} {peer_avg.get(k, 'N/A'):>20}")

---
## 7. Technical Analysis

In [ ]:
# ============================================================
# TECHNICAL ANALYSIS — 1 YEAR DETAILED
# ============================================================
# Use last ~18 months for detailed technicals
ta_data = price_data.iloc[-450:].copy()

# Calculate indicators
ta_data['EMA20'] = ta_data['Close'].ewm(span=20).mean()
ta_data['EMA50'] = ta_data['Close'].ewm(span=50).mean()
ta_data['EMA200'] = ta_data['Close'].ewm(span=200).mean()

# Bollinger Bands
ta_data['BB_mid'] = ta_data['Close'].rolling(20).mean()
bb_std = ta_data['Close'].rolling(20).std()
ta_data['BB_upper'] = ta_data['BB_mid'] + 2 * bb_std
ta_data['BB_lower'] = ta_data['BB_mid'] - 2 * bb_std

# MACD
ema12 = ta_data['Close'].ewm(span=12).mean()
ema26 = ta_data['Close'].ewm(span=26).mean()
ta_data['MACD'] = ema12 - ema26
ta_data['Signal'] = ta_data['MACD'].ewm(span=9).mean()
ta_data['MACD_Hist'] = ta_data['MACD'] - ta_data['Signal']

# RSI
delta = ta_data['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
ta_data['RSI'] = 100 - (100 / (1 + rs))

# ADX (Average Directional Index)
high = ta_data['High']
low = ta_data['Low']
close = ta_data['Close']
tr = pd.concat([high - low, abs(high - close.shift(1)), abs(low - close.shift(1))], axis=1).max(axis=1)
atr = tr.rolling(14).mean()

# Stochastic
low14 = ta_data['Low'].rolling(14).min()
high14 = ta_data['High'].rolling(14).max()
ta_data['Stoch_K'] = (ta_data['Close'] - low14) / (high14 - low14) * 100
ta_data['Stoch_D'] = ta_data['Stoch_K'].rolling(3).mean()

print("✅ Technical indicators calculated")

In [ ]:
# ============================================================
# MULTI-PANEL TECHNICAL CHART
# ============================================================
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    row_heights=[0.40, 0.12, 0.16, 0.16, 0.16],
                    subplot_titles=['Price Action & Bollinger Bands', 'Volume',
                                    'MACD', 'RSI (14)', 'Stochastic (14,3)'])

# 1. Candlestick + Bollinger Bands + EMAs
fig.add_trace(go.Candlestick(
    x=ta_data.index, open=ta_data['Open'], high=ta_data['High'],
    low=ta_data['Low'], close=ta_data['Close'], name='OHLC',
    increasing_line_color='#00d4aa', decreasing_line_color='#ff4757'
), row=1, col=1)

for col, color, name in [('EMA20', '#ffa502', 'EMA20'), ('EMA50', '#ff6348', 'EMA50'), ('EMA200', '#a29bfe', 'EMA200')]:
    fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data[col], name=name,
                             line=dict(color=color, width=1.2)), row=1, col=1)

fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['BB_upper'], name='BB Upper',
                         line=dict(color='rgba(255,255,255,0.3)', dash='dash'), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['BB_lower'], name='BB Lower',
                         fill='tonexty', fillcolor='rgba(255,255,255,0.05)',
                         line=dict(color='rgba(255,255,255,0.3)', dash='dash'), showlegend=False), row=1, col=1)

# 2. Volume
vol_colors = ['#00d4aa' if c >= o else '#ff4757' for c, o in zip(ta_data['Close'], ta_data['Open'])]
fig.add_trace(go.Bar(x=ta_data.index, y=ta_data['Volume'], name='Volume',
                     marker_color=vol_colors, showlegend=False), row=2, col=1)

# 3. MACD
macd_colors = ['#00d4aa' if v >= 0 else '#ff4757' for v in ta_data['MACD_Hist']]
fig.add_trace(go.Bar(x=ta_data.index, y=ta_data['MACD_Hist'], name='MACD Hist',
                     marker_color=macd_colors, showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['MACD'], name='MACD',
                         line=dict(color='#00d4aa', width=1.5)), row=3, col=1)
fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['Signal'], name='Signal',
                         line=dict(color='#ff6348', width=1.5)), row=3, col=1)

# 4. RSI
fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['RSI'], name='RSI',
                         line=dict(color='#ffa502', width=1.5)), row=4, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='rgba(255,71,87,0.5)', row=4, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='rgba(0,212,170,0.5)', row=4, col=1)

# 5. Stochastic
fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['Stoch_K'], name='%K',
                         line=dict(color='#4ecdc4', width=1.5)), row=5, col=1)
fig.add_trace(go.Scatter(x=ta_data.index, y=ta_data['Stoch_D'], name='%D',
                         line=dict(color='#ff6348', width=1.5)), row=5, col=1)
fig.add_hline(y=80, line_dash='dash', line_color='rgba(255,71,87,0.5)', row=5, col=1)
fig.add_hline(y=20, line_dash='dash', line_color='rgba(0,212,170,0.5)', row=5, col=1)

fig.update_layout(height=1100, template='plotly_dark', xaxis_rangeslider_visible=False,
                  title=f'{COMPANY} — Technical Analysis Dashboard',
                  legend=dict(orientation='h', yanchor='bottom', y=1.01, font=dict(size=10)))
fig.update_yaxes(title_text='Price (KRW)', row=1, col=1)
fig.show()

In [ ]:
# ============================================================
# TECHNICAL SIGNALS SUMMARY
# ============================================================
latest = ta_data.iloc[-1]
prev = ta_data.iloc[-2]

signals = []

# EMA signals
if latest['Close'] > latest['EMA200']:
    signals.append(('🟢', 'Price above 200 EMA', 'BULLISH — Long-term uptrend intact'))
else:
    signals.append(('🔴', 'Price below 200 EMA', 'BEARISH — Long-term downtrend'))

if latest['EMA50'] > latest['EMA200']:
    signals.append(('🟢', 'Golden Cross (50 > 200 EMA)', 'BULLISH'))
else:
    signals.append(('🔴', 'Death Cross (50 < 200 EMA)', 'BEARISH'))

if latest['Close'] > latest['EMA20']:
    signals.append(('🟢', 'Price above 20 EMA', 'Short-term bullish'))
else:
    signals.append(('🔴', 'Price below 20 EMA', 'Short-term bearish'))

# MACD
if latest['MACD'] > latest['Signal']:
    signals.append(('🟢', 'MACD above Signal line', 'BULLISH momentum'))
else:
    signals.append(('🔴', 'MACD below Signal line', 'BEARISH momentum'))

if latest['MACD_Hist'] > prev['MACD_Hist']:
    signals.append(('🟢', 'MACD Histogram increasing', 'Momentum strengthening'))
else:
    signals.append(('🔴', 'MACD Histogram decreasing', 'Momentum weakening'))

# RSI
rsi_val = latest['RSI']
if rsi_val > 70:
    signals.append(('🔴', f'RSI = {rsi_val:.1f}', 'OVERBOUGHT — Potential pullback'))
elif rsi_val < 30:
    signals.append(('🟢', f'RSI = {rsi_val:.1f}', 'OVERSOLD — Potential bounce'))
else:
    signals.append(('⚪', f'RSI = {rsi_val:.1f}', 'NEUTRAL'))

# Bollinger Bands
if latest['Close'] > latest['BB_upper']:
    signals.append(('🔴', 'Price above upper Bollinger Band', 'Overbought / Breakout'))
elif latest['Close'] < latest['BB_lower']:
    signals.append(('🟢', 'Price below lower Bollinger Band', 'Oversold / Breakdown'))
else:
    signals.append(('⚪', 'Price within Bollinger Bands', 'Normal range'))

# Stochastic
if latest['Stoch_K'] > 80:
    signals.append(('🔴', f'Stochastic %K = {latest["Stoch_K"]:.1f}', 'OVERBOUGHT'))
elif latest['Stoch_K'] < 20:
    signals.append(('🟢', f'Stochastic %K = {latest["Stoch_K"]:.1f}', 'OVERSOLD'))
else:
    signals.append(('⚪', f'Stochastic %K = {latest["Stoch_K"]:.1f}', 'NEUTRAL'))

print(f"{'='*75}")
print(f"  Technical Signals Summary — {ta_data.index[-1].date()}")
print(f"{'='*75}")
print(f"  Current Price: ₩{latest['Close']:,.0f}")
print(f"  20 EMA: ₩{latest['EMA20']:,.0f} | 50 EMA: ₩{latest['EMA50']:,.0f} | 200 EMA: ₩{latest['EMA200']:,.0f}")
print(f"{'='*75}")
for icon, signal, interpretation in signals:
    print(f"  {icon} {signal:<45} {interpretation}")

# Overall score
bull_count = sum(1 for s in signals if s[0] == '🟢')
bear_count = sum(1 for s in signals if s[0] == '🔴')
print(f"\n  📊 Technical Score: {bull_count} Bullish / {bear_count} Bearish / {len(signals)-bull_count-bear_count} Neutral")
if bull_count > bear_count + 2:
    print(f"  ➡️  Overall Technical Bias: BULLISH")
elif bear_count > bull_count + 2:
    print(f"  ➡️  Overall Technical Bias: BEARISH")
else:
    print(f"  ➡️  Overall Technical Bias: NEUTRAL / MIXED")

In [ ]:
# ============================================================
# DRAWDOWN ANALYSIS
# ============================================================
cummax = price_data['Close'].cummax()
drawdown = (price_data['Close'] / cummax - 1) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown, fill='tozeroy',
                         fillcolor='rgba(255,71,87,0.3)', line=dict(color='#ff4757', width=1),
                         name='Drawdown %'))

# Annotate worst drawdowns
worst_dd = drawdown.min()
worst_dd_date = drawdown.idxmin()
fig.add_annotation(x=worst_dd_date, y=worst_dd,
                   text=f'Max DD: {worst_dd:.1f}%', showarrow=True,
                   arrowhead=2, arrowcolor='white', font=dict(color='white'))

fig.update_layout(title=f'{COMPANY} — Drawdown Analysis (10Y)',
                  height=350, template='plotly_dark',
                  yaxis_title='Drawdown %')
fig.show()

---
## 8. Peer Comparison

In [ ]:
# ============================================================
# PEER COMPARISON — GLOBAL AUTO MANUFACTURERS
# ============================================================
peers = {
    'Hyundai': '005380.KS',
    'Toyota': 'TM',
    'Volkswagen': 'VWAGY',
    'GM': 'GM',
    'Ford': 'F',
    'Stellantis': 'STLA',
    'BMW': 'BMWYY',
    'Honda': 'HMC',
    'BYD': 'BYDDY'
}

peer_data = []
for name, ticker in peers.items():
    try:
        p = yf.Ticker(ticker)
        pi = p.info
        peer_data.append({
            'Company': name,
            'Mkt Cap ($B)': round(pi.get('marketCap', 0) / 1e9, 1),
            'Fwd P/E': round(pi.get('forwardPE', 0), 1) if pi.get('forwardPE') else None,
            'Trail P/E': round(pi.get('trailingPE', 0), 1) if pi.get('trailingPE') else None,
            'P/B': round(pi.get('priceToBook', 0), 2) if pi.get('priceToBook') else None,
            'EV/EBITDA': round(pi.get('enterpriseToEbitda', 0), 1) if pi.get('enterpriseToEbitda') else None,
            'EV/Rev': round(pi.get('enterpriseToRevenue', 0), 2) if pi.get('enterpriseToRevenue') else None,
            'Div Yield %': round(pi.get('dividendYield', 0) * 100, 2) if pi.get('dividendYield') else None,
            'Gross Margin %': round(pi.get('grossMargins', 0) * 100, 1) if pi.get('grossMargins') else None,
            'Op Margin %': round(pi.get('operatingMargins', 0) * 100, 1) if pi.get('operatingMargins') else None,
            'Net Margin %': round(pi.get('profitMargins', 0) * 100, 1) if pi.get('profitMargins') else None,
            'Beta': round(pi.get('beta', 0), 2) if pi.get('beta') else None,
            'ROE %': round(pi.get('returnOnEquity', 0) * 100, 1) if pi.get('returnOnEquity') else None,
            'Debt/Equity': round(pi.get('debtToEquity', 0) / 100, 2) if pi.get('debtToEquity') else None,
        })
    except Exception as e:
        print(f"  ⚠️ Could not fetch {name}: {e}")

peer_df = pd.DataFrame(peer_data).set_index('Company')

print(f"{'='*100}")
print(f"  Global Auto Manufacturer Peer Comparison")
print(f"{'='*100}")
print(peer_df.to_string())

In [ ]:
# ============================================================
# PEER COMPARISON RADAR CHART
# ============================================================
# Normalize metrics for radar chart (selected peers)
radar_peers = ['Hyundai', 'Toyota', 'GM', 'BYD', 'Volkswagen']
radar_metrics = ['Fwd P/E', 'P/B', 'Gross Margin %', 'Op Margin %', 'ROE %', 'Div Yield %']
radar_df = peer_df.loc[peer_df.index.isin(radar_peers), radar_metrics].dropna(axis=1, how='all')

fig = go.Figure()
colors = ['#00d4aa', '#ff6348', '#ffa502', '#4ecdc4', '#a29bfe']

for i, company in enumerate(radar_df.index):
    vals = radar_df.loc[company].values.tolist()
    vals.append(vals[0])  # close the polygon
    cats = radar_df.columns.tolist()
    cats.append(cats[0])
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats, name=company,
        line=dict(color=colors[i % len(colors)], width=2),
        fill='toself', opacity=0.3
    ))

fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, max(radar_df.max()) * 1.2])),
                  title=f'Peer Comparison — Radar Chart', height=550, template='plotly_dark')
fig.show()

In [ ]:
# ============================================================
# RELATIVE PRICE PERFORMANCE vs PEERS (1 YEAR)
# ============================================================
benchmark_tickers = {'Hyundai': '005380.KS', 'Toyota': 'TM', 'GM': 'GM', 'BYD': 'BYDDY', 'VW': 'VWAGY'}
perf_fig = go.Figure()
colors_list = ['#00d4aa', '#ff6348', '#ffa502', '#4ecdc4', '#a29bfe']

for i, (name, tick) in enumerate(benchmark_tickers.items()):
    try:
        hist = yf.Ticker(tick).history(period='1y')
        if len(hist) > 0:
            norm = (hist['Close'] / hist['Close'].iloc[0] - 1) * 100
            perf_fig.add_trace(go.Scatter(x=norm.index, y=norm, name=name,
                                          line=dict(color=colors_list[i], width=2.5 if name=='Hyundai' else 1.5)))
    except:
        pass

perf_fig.update_layout(title='Relative Price Performance vs Peers (1Y, % Change)',
                       height=450, template='plotly_dark', yaxis_title='Return %',
                       legend=dict(orientation='h', yanchor='bottom', y=-0.15))
perf_fig.add_hline(y=0, line_dash='dash', line_color='rgba(255,255,255,0.3)')
perf_fig.show()

---
## 9. Investment Scorecard & Thesis

In [ ]:
# ============================================================
# COMPREHENSIVE INVESTMENT SCORECARD
# ============================================================
def score_metric(value, thresholds, reverse=False):
    """Score a metric 1-5 based on thresholds. reverse=True means lower is better."""
    if value is None or np.isnan(value):
        return 3, '⚪'
    t = thresholds
    if reverse:
        if value <= t[0]: return 5, '🟢'
        elif value <= t[1]: return 4, '🟢'
        elif value <= t[2]: return 3, '🟡'
        elif value <= t[3]: return 2, '🟠'
        else: return 1, '🔴'
    else:
        if value >= t[3]: return 5, '🟢'
        elif value >= t[2]: return 4, '🟢'
        elif value >= t[1]: return 3, '🟡'
        elif value >= t[0]: return 2, '🟠'
        else: return 1, '🔴'

# Latest values
latest_yr = annual_data.index[-1]
gm = margins.loc[latest_yr, 'Gross Margin'] if latest_yr in margins.index else None
om = margins.loc[latest_yr, 'Operating Margin'] if latest_yr in margins.index else None
nm = margins.loc[latest_yr, 'Net Margin'] if latest_yr in margins.index else None
roe = bs_ratios.loc[latest_yr, 'ROE %'] if latest_yr in bs_ratios.index else None
roa = bs_ratios.loc[latest_yr, 'ROA %'] if latest_yr in bs_ratios.index else None
roic_val = bs_ratios.loc[latest_yr, 'ROIC %'] if latest_yr in bs_ratios.index else None
cr = bs_ratios.loc[latest_yr, 'Current Ratio'] if latest_yr in bs_ratios.index else None
de = bs_ratios.loc[latest_yr, 'Debt to Equity'] if latest_yr in bs_ratios.index else None
fcf_m = cf_data.loc[latest_yr, 'FCF Margin %'] if latest_yr in cf_data.index else None

scorecard = []

# Profitability
s, i = score_metric(gm, [10, 15, 20, 25])
scorecard.append(('Gross Margin', f'{gm:.1f}%' if gm else 'N/A', s, i, 'Auto avg ~18-22%'))
s, i = score_metric(om, [3, 5, 7, 10])
scorecard.append(('Operating Margin', f'{om:.1f}%' if om else 'N/A', s, i, 'Auto avg ~5-8%'))
s, i = score_metric(nm, [2, 4, 6, 8])
scorecard.append(('Net Margin', f'{nm:.1f}%' if nm else 'N/A', s, i, 'Auto avg ~3-6%'))
s, i = score_metric(roe, [5, 8, 12, 15])
scorecard.append(('ROE', f'{roe:.1f}%' if roe else 'N/A', s, i, 'Good >12%'))
s, i = score_metric(roic_val, [3, 5, 8, 12])
scorecard.append(('ROIC', f'{roic_val:.1f}%' if roic_val else 'N/A', s, i, 'Good >8%'))

# Financial Health
s, i = score_metric(cr, [0.8, 1.0, 1.2, 1.5])
scorecard.append(('Current Ratio', f'{cr:.2f}' if cr else 'N/A', s, i, 'Healthy >1.2'))
s, i = score_metric(de, [0.5, 1.0, 1.5, 2.0], reverse=True)
scorecard.append(('Debt/Equity', f'{de:.2f}' if de else 'N/A', s, i, 'Lower is better'))

# Cash Flow
s, i = score_metric(fcf_m, [1, 3, 5, 8])
scorecard.append(('FCF Margin', f'{fcf_m:.1f}%' if fcf_m else 'N/A', s, i, 'Good >5%'))

# Valuation
if pe_ratio:
    s, i = score_metric(pe_ratio, [5, 8, 12, 18], reverse=True)
    scorecard.append(('P/E Ratio', f'{pe_ratio:.1f}x', s, i, 'Auto avg 8-12x'))
if ev_ebitda:
    s, i = score_metric(ev_ebitda, [4, 6, 8, 12], reverse=True)
    scorecard.append(('EV/EBITDA', f'{ev_ebitda:.1f}x', s, i, 'Auto avg 4-7x'))
if fcf_yield:
    s, i = score_metric(fcf_yield, [2, 4, 6, 8])
    scorecard.append(('FCF Yield', f'{fcf_yield:.1f}%', s, i, 'Good >5%'))

# Growth
s, i = score_metric(rev_cagr*100, [2, 5, 8, 12])
scorecard.append(('Revenue CAGR', f'{rev_cagr*100:.1f}%', s, i, f'{n_years}yr CAGR'))

# DCF
if upside > 20:
    s, i = 5, '🟢'
elif upside > 10:
    s, i = 4, '🟢'
elif upside > -10:
    s, i = 3, '🟡'
elif upside > -20:
    s, i = 2, '🟠'
else:
    s, i = 1, '🔴'
scorecard.append(('DCF Upside', f'{upside:+.1f}%', s, i, 'vs Current Price'))

print(f"{'='*85}")
print(f"  {COMPANY} — Investment Scorecard")
print(f"{'='*85}")
print(f"  {'Metric':<22} {'Value':>12} {'Score':>8} {'':>3} {'Benchmark':>20}")
print(f"  {'-'*75}")

total_score = 0
for metric, value, score_val, icon, benchmark in scorecard:
    total_score += score_val
    bars = '█' * score_val + '░' * (5 - score_val)
    print(f"  {icon} {metric:<20} {value:>12} {bars} {score_val}/5  {benchmark:>20}")

avg_score = total_score / len(scorecard)
print(f"\n  {'='*75}")
print(f"  ⭐ OVERALL SCORE: {avg_score:.1f} / 5.0 ({total_score}/{len(scorecard)*5})")

if avg_score >= 4.0:
    verdict = '🟢 STRONG BUY'
elif avg_score >= 3.5:
    verdict = '🟢 BUY'
elif avg_score >= 3.0:
    verdict = '🟡 HOLD / ACCUMULATE'
elif avg_score >= 2.5:
    verdict = '🟠 HOLD / CAUTIOUS'
else:
    verdict = '🔴 SELL / AVOID'

print(f"  📋 VERDICT: {verdict}")
print(f"  {'='*75}")

In [ ]:
# ============================================================
# INVESTMENT THESIS SUMMARY
# ============================================================
print(f"""
{'='*80}
  INVESTMENT THESIS — {COMPANY}
{'='*80}

  BULL CASE:
  ━━━━━━━━━
  • Global EV transition: Hyundai/Kia's IONIQ lineup gaining significant share
    in the EV market, with strong consumer reception and competitive pricing
  • Premium brand elevation: Genesis brand pushing ASPs higher and improving
    overall margin mix toward luxury segment economics
  • Manufacturing excellence: World-class production efficiency with expanding
    US manufacturing footprint (Georgia EV plant) securing IRA incentives
  • Hydrogen leadership: Long-term optionality via hydrogen fuel cell technology
    leadership (NEXO, commercial vehicles) as a diversification play
  • Valuation discount: Trading at significant discount to global peers on
    P/E and EV/EBITDA, reflecting Korean conglomerate discount
  • Balance sheet strength: Strong cash generation funding both growth 
    investments and shareholder returns simultaneously

  BEAR CASE:
  ━━━━━━━━━
  • Chinese EV competition: BYD and other Chinese OEMs aggressively expanding
    globally with cost advantages, particularly in emerging markets
  • Korean discount persistence: Chaebol governance structure and complex
    cross-holdings may continue to suppress valuation multiples
  • Currency exposure: Significant KRW/USD and KRW/EUR sensitivity can
    impact reported earnings and competitiveness
  • EV margin compression: Battery costs still elevated and EV margins
    structurally lower than ICE, creating transition period pressure
  • Geopolitical risks: Trade tensions, tariff uncertainty, and supply
    chain vulnerabilities in semiconductor/battery raw materials
  • Cyclicality: Auto industry inherently cyclical; macroeconomic slowdown
    could pressure volumes and pricing power simultaneously

  KEY CATALYSTS TO WATCH:
  ━━━━━━━━━━━━━━━━━━━━━━
  • Global EV sales trajectory and market share gains
  • US/EU tariff policy developments
  • Genesis brand performance and geographic expansion
  • Georgia EV manufacturing plant ramp-up and IRA credit eligibility
  • Software-defined vehicle (SDV) platform development progress
  • Shareholder return policy evolution (buybacks, dividend growth)
  • Robotics & Urban Air Mobility (Boston Dynamics, Supernal)
{'='*80}
""")

In [ ]:
# ============================================================
# DIVIDEND ANALYSIS
# ============================================================
divs = stock.dividends
if divs is not None and len(divs) > 0:
    divs.index = divs.index.tz_localize(None)
    annual_divs = divs.resample('YE').sum()
    annual_divs.index = annual_divs.index.year
    
    print(f"{'='*50}")
    print(f"  Dividend History (KRW per share)")
    print(f"{'='*50}")
    for yr, div in annual_divs.items():
        if div > 0:
            print(f"  {yr}: ₩{div:,.0f}")
    
    # Payout ratio
    if len(annual_divs) > 0 and len(annual_data) > 0:
        common_yrs = sorted(set(annual_divs.index) & set(annual_data.index))
        if common_yrs:
            print(f"\n  Payout Ratios:")
            for yr in common_yrs:
                eps = annual_data.loc[yr, 'Net Income'] / shares_out if yr in annual_data.index else 0
                if eps > 0:
                    payout = annual_divs[yr] / eps * 100
                    print(f"  {yr}: {payout:.1f}%")
    
    fig = go.Figure()
    fig.add_trace(go.Bar(x=annual_divs.index.astype(str), y=annual_divs.values,
                         marker_color='#00d4aa', name='Dividend/Share'))
    fig.update_layout(title=f'{COMPANY} — Annual Dividend per Share (KRW)',
                      height=350, template='plotly_dark', yaxis_title='KRW per Share')
    fig.show()
else:
    print("No dividend data available")

In [ ]:
# ============================================================
# EARNINGS QUALITY — OCF vs NET INCOME
# ============================================================
fig = go.Figure()

years_str = annual_data.index.astype(str)
fig.add_trace(go.Bar(x=years_str, y=annual_data['Net Income']/T, name='Net Income',
                     marker_color='#4ecdc4', opacity=0.8))
fig.add_trace(go.Bar(x=years_str, y=cf_data['Operating CF']/T, name='Operating Cash Flow',
                     marker_color='#00d4aa', opacity=0.8))
fig.add_trace(go.Bar(x=years_str, y=cf_data['Free Cash Flow']/T, name='Free Cash Flow',
                     marker_color='#ffa502', opacity=0.8))

fig.update_layout(title=f'{COMPANY} — Earnings Quality: Net Income vs Cash Flows (KRW T)',
                  height=400, template='plotly_dark', barmode='group',
                  yaxis_title='KRW Trillions')
fig.show()

In [ ]:
# ============================================================
# FINAL DISCLAIMER
# ============================================================
print(f"""
{'='*80}
  ⚠️  DISCLAIMER
{'='*80}
  This analysis is for informational and educational purposes only.
  It does NOT constitute financial advice, investment recommendations,
  or an offer to buy or sell securities.
  
  Key limitations:
  • Data sourced from Yahoo Finance; may contain errors or omissions
  • DCF models are highly sensitive to growth and discount rate assumptions
  • Past performance does not guarantee future results
  • Technical analysis is backward-looking and not predictive
  • This does not account for geopolitical risks, regulatory changes,
    or company-specific events not reflected in financial statements
  
  Always conduct your own research and consult with qualified financial
  advisors before making investment decisions.
{'='*80}
""")